# Public Final Policy Outputs for RRWH

This notebook reads the public workflow outputs and produces a district-level summary table for rainwater harvesting policy reporting.

## Inputs

- `public_rrwh_workflow/config/paths.yaml`
- A rainfall output raster such as `annual_RWH.nc` or `annual_rainfall_chirps_native.nc`
- An administrative boundary file in `paths['data']['admin_boundaries']`

## Outputs

- `output/tables/district_rwh_summary.csv`
- Optional plots or maps can be added later using the public plotting helpers

In [ ]:
from pathlib import Path
import os
import sys

import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
from rasterstats import zonal_stats

WORKFLOW_ROOT = Path(__file__).resolve().parents[1] if '__file__' in globals() else Path.cwd() / 'public_rrwh_workflow'
SRC_DIR = WORKFLOW_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from preprocessing import load_paths

paths = load_paths(WORKFLOW_ROOT / 'config' / 'paths.yaml')
maps_dir = Path(paths['output']['maps'])
tables_dir = Path(paths['output']['tables'])
tables_dir.mkdir(parents=True, exist_ok=True)

rwh_candidates = [
    maps_dir / 'annual_RWH.nc',
    maps_dir / 'annual_rainfall_chirps_native.nc',
]
rwh_path = next((path for path in rwh_candidates if path.exists()), None)
if rwh_path is None:
    raise FileNotFoundError(f'No RWH raster found in {maps_dir}')

admin_path = Path(paths['data']['admin_boundaries'])
if not admin_path.exists():
    raise FileNotFoundError(f'Admin boundary file not found: {admin_path}')

print(f'Loading rainfall output from: {rwh_path}')
rwh = xr.open_dataset(rwh_path)
if '__xarray_dataarray_variable__' in rwh:
    rwh = rwh['__xarray_dataarray_variable__']

gdf = gpd.read_file(admin_path)
if gdf.crs is None:
    gdf = gdf.set_crs('EPSG:4326')

transform = rwh.rio.transform()
values = rwh.values

print('Calculating zonal statistics...')
zs = zonal_stats(gdf, values, affine=transform, stats=['sum', 'mean', 'count'], nodata=np.nan)
gdf['total_rwh_liters'] = [entry['sum'] for entry in zs]
gdf['mean_rwh_pixel'] = [entry['mean'] for entry in zs]
gdf['pixel_count'] = [entry['count'] for entry in zs]

table_path = tables_dir / 'district_rwh_summary.csv'
gdf.drop(columns='geometry').to_csv(table_path, index=False)
print(f'Saved district summary to {table_path}')

sort_cols = [col for col in ['total_rwh_liters', 'mean_rwh_pixel'] if col in gdf.columns]
print(gdf.sort_values(by=sort_cols, ascending=False).head())